# Multi-Step Reasoning And ReAct

ReAct is a core technique in artificial intelligence where a model alternates between three steps:

- **Reason (Thought):** The model analyzes the problem, tracks context, and decides the next useful action.
- **Act (Tool use):** A local tool performs the action (e.g. calling an API, running a code block, or searching the web using specific inputs).
- **Observe (Result):** The system feeds the raw output or result of that action back to the model, grounding the next phase of thinking in actual facts.

In production, this pattern appears in support and data assistants, coding agents, incident triage, and workflow automation. The inspectable trace should contain selected actions, tool observations, and a concise final rationale, not the model's private chain-of-thought.


## Learning Objectives

By the end of this notebook, you should be able to:

- Design a JSON action contract that asks a model for its next step.
- Execute only allow-listed local tools in response to a model action.
- Store observations after each step to build an inspectable trace.
- Produce a concise final answer from accumulated observations.
- Avoid asking a model to reveal private chain-of-thought.


## Reasoning And Retrieval Context

Multi-step prompting often combines model reasoning with retrieved or tool-provided context. RAG is not the same as ReAct, but the diagram below shows the same central idea: external information is retrieved, inserted into the model context, and used to ground the final answer.

<div align="center">
  <img src="assets/03-rag-diagram.png" alt="Retrieval-augmented generation diagram" width="700" style="max-width: 100%; height: auto;" />
  <p><em>Retrieval-augmented generation diagram. Source: <a href="https://commons.wikimedia.org/wiki/File:RAG_diagram.svg" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>

The visual reasoning example below is a useful reminder that modern LLM workflows increasingly combine language instructions with multimodal evidence. Even though this notebook uses text-only local tools, the same orchestration pattern appears when models inspect images, documents, search results, or application state.

<div align="center">
  <img src="assets/03-gemini-reasoning-example.png" alt="Multimodal reasoning example" width="650" style="max-width: 100%; height: auto;" />
  <p><em>Multimodal reasoning example. Source: <a href="https://commons.wikimedia.org/wiki/File:Gemini_image_understanding_and_reasoning.png" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>

## Provider And Tool-Use References

This notebook uses a simple JSON action contract rather than provider-native tool calls so the loop is easy to inspect. Provider-native function calling follows the same idea: define tools, let the model choose a tool call, execute the tool in your code, then return the result.

To learn more about these, visit:
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)
- [Mistral Function Calling](https://docs.mistral.ai/capabilities/function_calling)
- [LangChain Tool Calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling)
- [PydanticAI](https://ai.pydantic.dev/)


## Shared Provider Helpers


In [ ]:
import json
from typing import Any

from openai import BadRequestError

from prompt_helpers import chat, client_and_model

## Local Tools

A **tool** is a plain Python function the model can request by name, so it can affect something beyond generating text. This notebook uses two small, hardcoded dictionaries in place of a real ticketing system and policy knowledge base.


In [ ]:
SUPPORT_TICKETS = {
    "T-104": {
        "customer": "Acme Analytics",
        "plan": "Team",
        "issue": "MFA reset after phone replacement",
        "status": "waiting_for_identity_verification",
    },
    "T-205": {
        "customer": "Northwind Labs",
        "plan": "Enterprise",
        "issue": "CSV export returns intermittent 500 errors",
        "status": "engineering_investigating",
    },
}

POLICY_SNIPPETS = {
    "mfa_reset": "For MFA resets, verify identity with two account recovery signals before disabling MFA.",
    "bug_escalation": "Escalate repeatable 5xx errors with request IDs, timestamps, and affected workspace.",
}

Tools should be explicit. The functions below wrap the local data into two callable actions: `lookup_ticket` and `lookup_policy`. The `TOOLS` dictionary maps each action's name to its function, so the model can select an action by name rather than by writing code. These are the only two actions the model will be able to request.


In [ ]:
from collections.abc import Callable


def lookup_ticket(ticket_id: str) -> dict[str, str]:
    """Return support-ticket details from the local demo store."""

    return SUPPORT_TICKETS.get(ticket_id, {"error": "unknown ticket"})


def lookup_policy(topic: str) -> dict[str, str]:
    """Return the operational policy text for a known support topic."""

    return {"topic": topic, "policy": POLICY_SNIPPETS.get(topic, "No policy found.")}


TOOLS: dict[str, Callable[..., dict[str, str]]] = {
    "lookup_ticket": lookup_ticket,
    "lookup_policy": lookup_policy,
}

## Action Contract

The model must choose from a constrained JSON shape: `action` names the next step, `arguments` supplies its inputs, and `final_answer` holds the answer once the model is done. This gives the loop a stable interface even when the natural-language task varies. The system prompt also names each tool's argument keys, because the model cannot see the Python function signatures and would otherwise guess them. It also hides the model's reasoning, so only the chosen action and the final answer are visible.


In [ ]:
ACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "action": {
            "type": "string",
            "enum": ["lookup_ticket", "lookup_policy", "final_answer"],
        },
        "arguments": {"type": "object"},
        "final_answer": {"type": ["string", "null"]},
    },
    "required": ["action", "arguments", "final_answer"],
    "additionalProperties": False,
}

SYSTEM_PROMPT = f"""You help a support operator decide the next step.

Return only JSON matching this schema:
{json.dumps(ACTION_SCHEMA, indent=2)}

Rules:
- Use lookup_ticket with arguments {{"ticket_id": "<ticket id>"}} when you need ticket details.
- Use lookup_policy with arguments {{"topic": "mfa_reset"}} or {{"topic": "bug_escalation"}} when you need a policy.
- Use final_answer only when you have enough information.
- Do not reveal hidden chain-of-thought; keep final answers concise."""

## Ask The Model For The Next Action

Before wiring up a live call, it helps to have a deterministic stand-in so the loop can still run without an API key. The function below hardcodes a fixed sequence: ticket lookup, then policy lookup, then a final answer, mimicking what a real decision would look like.


In [ ]:
def dry_react_action(
    task: str, observations: list[dict[str, object]]
) -> dict[str, object]:
    """Deterministic fallback so the loop is runnable without an API key."""

    if not observations:
        return {
            "action": "lookup_ticket",
            "arguments": {"ticket_id": "T-104"},
            "final_answer": None,
        }
    if len(observations) == 1:
        return {
            "action": "lookup_policy",
            "arguments": {"topic": "mfa_reset"},
            "final_answer": None,
        }
    return {
        "action": "final_answer",
        "arguments": {},
        "final_answer": "Verify two recovery signals before disabling MFA for ticket T-104.",
    }

The function below is the real decision-maker: it sends the task and the observations collected so far to the model, or falls back to the function above when no provider is configured. Malformed responses and provider JSON rejections become error observations so the loop can retry within its step budget. Notice that observations are data, not instructions. The system message still controls the action contract.


In [ ]:
def next_action(task: str, observations: list[dict[str, object]]) -> dict[str, object]:
    """Request the next action from the model and parse it as JSON."""

    client, _, _, dry_reason = client_and_model()
    if client is None:
        print(dry_reason)
        return dry_react_action(task, observations)

    user_prompt = f"""Task: {task}

Observations so far:
{json.dumps(observations, indent=2)}"""
    try:
        raw = chat(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.1,
            response_format={"type": "json_object"},
        )
        action = json.loads(raw)
    except (BadRequestError, json.JSONDecodeError):
        action = None
    if not isinstance(action, dict) or not isinstance(action.get("action"), str):
        # execute_action turns an unknown action name into an error observation.
        return {"action": "invalid_response", "arguments": {}, "final_answer": None}
    return action

## Execute Only Allow-Listed Tools

This is the security boundary. The model can request `lookup_ticket`, but it cannot execute arbitrary Python or invent new tool names. Anything outside `TOOLS` is rejected, and a call with the wrong argument names comes back as an error observation instead of crashing the loop, so the model can correct itself on the next step. The function always returns a plain dictionary, which will become the observation record that gets appended to the trace and fed back into the next step.


In [ ]:
def execute_action(action: dict[str, object]) -> dict[str, object]:
    """Run an allow-listed tool call and return an observation record."""

    action_name = str(action.get("action"))
    if action_name == "final_answer":
        answer = action.get("final_answer")
        if not isinstance(answer, str) or not answer.strip():
            return {"type": "error", "content": "Final answer must be non-empty text."}
        return {"type": "final", "content": answer}

    if action_name not in TOOLS:
        return {"type": "error", "content": f"Blocked unknown tool: {action_name}"}

    raw_arguments = action.get("arguments") or {}
    if not isinstance(raw_arguments, dict):
        return {"type": "error", "content": "Tool arguments must be an object."}

    # Convert JSON values to strings because the two demo tools accept string arguments.
    arguments = {str(key): str(value) for key, value in raw_arguments.items()}
    tool = TOOLS[action_name]
    try:
        result = tool(**arguments)
    except TypeError as error:
        # Wrong argument names become an observation the model can correct on its next step.
        return {"type": "error", "content": f"Bad arguments for {action_name}: {error}"}
    return {
        "type": "tool_result",
        "tool": action_name,
        "arguments": arguments,
        "content": result,
    }

## Run The Loop

Each iteration is one full Reason–Act–Observe cycle: `next_action` reasons about what to do next, `execute_action` acts on it, and the result is observed and appended to `observations` before the loop reasons again.

The loop stops when the model returns `final_answer` or when it reaches the step limit. Having a step limit protects you from infinite loops and runaway API usage. The returned `trace` is the full step-by-step history; `final` is the model's final answer, or a note that the loop hit the step limit.


In [ ]:
def run_react(task: str, max_steps: int = 4) -> dict[str, Any]:
    """Run a small ReAct loop and keep the trace inspectable."""

    if max_steps < 1:
        raise ValueError("max_steps must be at least 1.")

    observations: list[dict[str, Any]] = []
    for step in range(1, max_steps + 1):
        action = next_action(task, observations)
        observation = execute_action(action)
        observations.append(
            {"step": step, "action": action, "observation": observation}
        )
        if observation["type"] == "final":
            break
    last = observations[-1]["observation"]
    return {
        "task": task,
        "trace": observations,
        "final": last["content"]
        if last["type"] == "final"
        else f"No final answer within {max_steps} steps.",
    }


result = run_react("For ticket T-104, decide the next safe support action.")
print(json.dumps(result, indent=2))

## Final Answer Discipline

For multi-step work, keep two outputs separate:

- the trace, which is for developers and auditors,
- the final answer, which is for the end user or operator.

The final answer should cite observations or tool results, not hidden reasoning.


In [ ]:
print("Final answer:")
print(result["final"])

print("\nTrace summary:")
for item in result["trace"]:
    action = item["action"]["action"]
    observation_type = item["observation"]["type"]
    print(f"Step {item['step']}: {action} -> {observation_type}")

## Practice

Add a new tool named `lookup_customer_plan(customer: str)` that looks up a customer in `CUSTOMER_PLANS` below and returns their plan details. Then update the action prompt, including the tool's argument name, so the model can use it before producing a final answer.

Keep the allow-list boundary intact: the tool name must appear in `TOOLS`, arguments must be a JSON object, and the loop should still stop after `max_steps`.


In [ ]:
CUSTOMER_PLANS = {
    "Acme Analytics": {
        "plan": "Team",
        "mfa_recovery": "identity verification required",
    },
}

# @TODO: Add a new tool `lookup_customer_plan` and update the system prompt.

### Reference Answer

A tool becomes available to the model only after two changes: implement and allow-list the function, then advertise the new action in the JSON contract and system prompt. The execution boundary remains unchanged because `execute_action` still rejects names outside `TOOLS`.


In [ ]:
def lookup_customer_plan(customer: str) -> dict[str, str]:
    """Return plan-level support context for an allow-listed customer."""

    return CUSTOMER_PLANS.get(customer, {"error": "unknown customer"})


TOOLS["lookup_customer_plan"] = lookup_customer_plan
print(
    execute_action(
        {"action": "lookup_customer_plan", "arguments": {"customer": "Acme Analytics"}}
    )
)

Now extend only the action contract. The rules name each tool's argument keys, as in the original prompt, so the model does not have to guess `customer`. In a live call, the model can select `lookup_customer_plan`; the same allow-listed dispatcher executes it and the existing `max_steps` argument still limits the loop.


In [ ]:
extended_schema = {
    **ACTION_SCHEMA,
    "properties": {
        **ACTION_SCHEMA["properties"],
        "action": {
            "type": "string",
            "enum": [
                "lookup_ticket",
                "lookup_policy",
                "lookup_customer_plan",
                "final_answer",
            ],
        },
    },
}
SYSTEM_PROMPT = f"""You help a support operator decide the next step.

Return only JSON matching this schema:
{json.dumps(extended_schema, indent=2)}

Rules:
- Use lookup_ticket with arguments {{"ticket_id": "<ticket id>"}} when you need ticket details.
- Use lookup_policy with arguments {{"topic": "mfa_reset"}} or {{"topic": "bug_escalation"}} when you need a policy.
- Use lookup_customer_plan with arguments {{"customer": "<customer name>"}} when plan context affects the safe next action.
- Use final_answer only when you have enough information.
- Do not reveal hidden chain-of-thought; keep final answers concise."""
print(SYSTEM_PROMPT)

## Further Reading

- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) - Yao et al., 2022
- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401) - Lewis et al., 2020
